# H3 linetrace algorithm

Step-by-step visualization of polyline → H3 cell path conversion, matching vgrid [`polyline2h3`](https://github.com/opengeoshub/vgrid/blob/main/vgrid/conversion/vector2dggs/vector2h3.py). Same multipolyline style as [`02_a5_linetrace.ipynb`](02_a5_linetrace.ipynb).

For each line part, on every consecutive vertex pair:

1. `h3.latlng_to_cell` at segment start and end
2. `h3.grid_path_cells(start_cell, end_cell)` for the segment path
3. Merge paths (drop only immediate duplicates at segment joins)

Input: [`multipolyline.geojson`](https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolyline.geojson) — **5 features**, **6 line parts**.

## Install necessary packages

In [1]:
# %pip install vgrid geopandas matplotlib imageio pillow
# optional for MP4:
%pip install imageio imageio-ffmpeg

Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""Step-by-step polyline2h3 path animation (multipolyline.geojson)."""
from pathlib import Path

import geopandas as gpd
import h3
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
from shapely.geometry import LineString, MultiLineString

from vgrid.conversion.dggs2geo.h32geo import h32geo
from vgrid.conversion.vector2dggs.vector2h3 import polyline2h3

URL = "https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolyline.geojson"
RESOLUTION = 11
OUT_GIF = "polyline2h3.gif"
OUT_MP4 = "polyline2h3.mp4"
FRAME_EVERY_N_PATH = 5  # 1 = frame per cell; use 5+ for long multipolyline runs
DPI = 120
PART_COLORS = ["#1f4e79", "#c55a11", "#2e7d32", "#b71c1c", "#6a1b9a", "#4e342e"]


def cell_patches(cell_polys, facecolor, edgecolor, alpha=0.55, lw=0.4):
    patches = []
    for poly in cell_polys:
        if poly is None or poly.is_empty:
            continue
        patches.append(MplPolygon(list(poly.exterior.coords), closed=True))
    return PatchCollection(
        patches, facecolor=facecolor, edgecolor=edgecolor, alpha=alpha, linewidths=lw
    )


def polylines_from_feature(feature):
    if feature.geom_type == "LineString":
        return [feature]
    if feature.geom_type == "MultiLineString":
        return list(feature.geoms)
    return []


def polylines_from_gdf(gdf):
    parts = []
    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue
        parts.extend(polylines_from_feature(geom))
    return parts


def feature_from_gdf(gdf):
    parts = polylines_from_gdf(gdf)
    if not parts:
        raise ValueError("No line geometries found in input GeoJSON")
    if len(parts) == 1:
        return parts[0]
    return MultiLineString(parts)


def part_color(part_index):
    return PART_COLORS[part_index % len(PART_COLORS)]


def render_frame(
    parts,
    segment_line,
    path_polys,
    title,
    path,
    resolution,
    current_poly=None,
    endpoint_polys=None,
    active_part=None,
):
    fig, ax = plt.subplots(figsize=(8, 8))
    minx, miny, maxx, maxy = MultiLineString(parts).bounds
    pad = max(maxx - minx, maxy - miny) * 0.08 or 0.01
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    for i, line in enumerate(parts):
        color = part_color(i)
        lw = 3.5 if active_part == i else 2.0
        alpha = 1.0 if active_part is None or active_part == i else 0.45
        gpd.GeoSeries([line]).plot(
            ax=ax, facecolor="none", edgecolor=color, lw=lw, alpha=alpha
        )
    if segment_line is not None:
        gpd.GeoSeries([segment_line]).plot(
            ax=ax, facecolor="none", edgecolor="#1f77b4", lw=3.5
        )

    visited = list(path_polys) if path_polys else []
    if current_poly is not None:
        visited = [
            p
            for p in visited
            if p is not current_poly and not p.equals(current_poly)
        ]
    if visited:
        ax.add_collection(cell_patches(visited, "#2ca02c", "#1a5f1a", alpha=0.45))

    if endpoint_polys:
        ax.add_collection(
            cell_patches(endpoint_polys, "#d62728", "#8b0000", alpha=0.7)
        )

    if current_poly is not None:
        ax.add_collection(
            cell_patches([current_poly], "#ffcc00", "#cc8800", alpha=0.9, lw=2.5)
        )

    # Fixed legend on every frame so GIF/MP4 frames share identical dimensions.
    ax.plot([], [], color="#ffcc00", lw=4, label="current cell")
    ax.plot([], [], color="#2ca02c", lw=4, label="path so far")
    ax.plot([], [], color="#d62728", lw=4, label="segment endpoints")
    ax.legend(loc="upper right", fontsize=8)
    ax.text(
        0.02,
        0.98,
        f"H3 resolution: {resolution}",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.9),
        zorder=6,
    )
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.08)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)


def polyline2h3_with_frames(parts, feature, resolution, frame_dir):
    """Mirror polyline2h3 in vector2h3.py with frame capture for all parts."""
    frame_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    idx = 0

    if not parts:
        return frames, []

    ordered_cells = []
    path_polys = []

    def snap(
        segment_line,
        title,
        current=None,
        endpoints=None,
        active_part=None,
    ):
        nonlocal idx
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(
            parts,
            segment_line,
            path_polys,
            title,
            p,
            resolution,
            current_poly=current,
            endpoint_polys=endpoints,
            active_part=active_part,
        )
        frames.append(p)
        idx += 1

    n_parts = len(parts)
    n_verts = sum(len(list(line.coords)) for line in parts)
    n_closed = sum(1 for line in parts if line.coords[0] == line.coords[-1])
    snap(
        None,
        f"1. Input res {resolution} ({n_parts} parts, {n_verts} vertices, "
        f"{n_closed} closed loop{'s' if n_closed != 1 else ''})",
    )
    snap(None, "2. All parts (distinct colors)")

    for part_i, polyline in enumerate(parts, start=1):
        coords = list(polyline.coords)
        if len(coords) < 2:
            continue

        part_cells_start = len(ordered_cells)
        is_closed = coords[0] == coords[-1]
        loop_note = " [closed loop]" if is_closed else ""

        for seg_i in range(len(coords) - 1):
            start_x, start_y = coords[seg_i]
            end_x, end_y = coords[seg_i + 1]
            segment_line = LineString([(start_x, start_y), (end_x, end_y)])

            start_cell = h3.latlng_to_cell(start_y, start_x, resolution)
            end_cell = h3.latlng_to_cell(end_y, end_x, resolution)
            start_poly = h32geo(start_cell)
            end_poly = h32geo(end_cell)

            snap(
                segment_line,
                f"3. Part {part_i}/{n_parts}{loop_note} seg {seg_i + 1}/"
                f"{len(coords) - 1}: vertex {seg_i} → {seg_i + 1}",
                endpoints=[start_poly, end_poly],
                active_part=part_i - 1,
            )

            try:
                segment_cells = list(h3.grid_path_cells(start_cell, end_cell))
            except Exception:
                segment_cells = []

            snap(
                segment_line,
                f"3b. grid_path_cells ({len(segment_cells)} cells)",
                endpoints=[start_poly, end_poly],
                active_part=part_i - 1,
            )

            for step, cell_id in enumerate(segment_cells, start=1):
                if ordered_cells and ordered_cells[-1] == cell_id:
                    continue
                ordered_cells.append(cell_id)
                cell_poly = h32geo(cell_id)
                if cell_poly is None or cell_poly.is_empty:
                    continue
                path_polys.append(cell_poly)
                if step % FRAME_EVERY_N_PATH == 0 or step == len(segment_cells):
                    snap(
                        segment_line,
                        f"4. Part {part_i} seg {seg_i + 1} step "
                        f"{step}/{len(segment_cells)}: {cell_id} "
                        f"({len(ordered_cells)} total)",
                        current=cell_poly,
                        endpoints=[start_poly, end_poly],
                        active_part=part_i - 1,
                    )

        snap(
            None,
            f"5. Part {part_i} done ({len(ordered_cells) - part_cells_start} cells)",
            active_part=part_i - 1,
        )

    snap(None, f"6. Complete path ({len(ordered_cells)} cells)")

    rows = polyline2h3(feature, resolution)
    from_polyline2h3 = [row["h3"] for row in rows]
    if set(from_polyline2h3) != set(ordered_cells):
        print(
            "Warning: cell set differs from polyline2h3:",
            len(ordered_cells),
            "vs",
            len(from_polyline2h3),
        )
    elif len(from_polyline2h3) != len(ordered_cells):
        print(
            f"Note: same unique cells; polyline2h3 has "
            f"{len(from_polyline2h3)} rows "
            f"({len(from_polyline2h3) - len(ordered_cells)} duplicates)"
        )

    return frames, ordered_cells


def main():
    gdf = gpd.read_file(URL)
    parts = polylines_from_gdf(gdf)
    feature = feature_from_gdf(gdf)
    print(f"Loaded {len(gdf)} feature(s), {len(parts)} line part(s)")

    frame_dir = Path("_polyline2h3_frames")
    frames, cell_ids = polyline2h3_with_frames(
        parts, feature, RESOLUTION, frame_dir
    )
    imageio.mimsave(OUT_GIF, [imageio.imread(f) for f in frames], duration=0.9)
    print(f"Wrote {OUT_GIF} ({len(frames)} frames, {len(cell_ids)} final cells)")
    try:
        writer = imageio.get_writer(OUT_MP4, fps=1.2)
        for f in frames:
            writer.append_data(imageio.imread(f))
        writer.close()
        print(f"Wrote {OUT_MP4}")
    except Exception as e:
        print(f"MP4 skipped ({e}). GIF is enough.")


if __name__ == "__main__":
    main()